# Which outputs an edit makes stale

A notebook does not record which cells depend on which. After you edit a cell and run it,
the outputs below it still come from the old version, and nothing marks the ones that are
now out of date. Here you work that out from a summary of each cell: the names it assigns
and the names it reads.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
from collections import Counter

# Each cell, top to bottom: the names it assigns ("defines") and the names it reads ("uses").
cells = [
    {"defines": {"np", "pd"}, "uses": set()},                  # 0  imports
    {"defines": {"df"}, "uses": {"pd"}},                        # 1  df = pd.read_csv(url)
    {"defines": {"lr"}, "uses": set()},                         # 2  lr = 0.1
    {"defines": {"X", "y"}, "uses": {"df", "np"}},              # 3  X, y = features and labels from df
    {"defines": {"X"}, "uses": {"X", "np"}},                    # 4  X = X / np.abs(X).max()
    {"defines": {"w"}, "uses": {"X", "y", "lr", "np"}},         # 5  w = gradient descent with lr
    {"defines": {"lr"}, "uses": set()},                         # 6  lr = 0.01
    {"defines": {"w_small"}, "uses": {"X", "y", "lr", "np"}},   # 7  w_small = the same with this lr
    {"defines": set(), "uses": {"w", "w_small"}},               # 8  print(w, w_small)
    {"defines": set(), "uses": {"df"}},                         # 9  print(len(df))
]
print(len(cells), "cells")
print("cells assigning each name:", dict(Counter(name for cell in cells for name in cell["defines"])))

## Part 1 (coding task): The cells an edit reaches

Write `stale_after_edit(cells, edited)`. Cell `edited` has just been changed. Return, in
increasing order, the indices of the cells below it whose output a run from the top could now
change.

In a run from the top, the value a cell reads for a name is the one assigned by the nearest
cell above it that defines the name. A cell's output can change when it reads a name whose
value comes from the edited cell, or from another cell whose output can change.

In [ ]:
def stale_after_edit(cells, edited):
    # TODO
    raise NotImplementedError

In [ ]:
r = stale_after_edit(cells, 2)
print("after editing cell 2:", r)
assert isinstance(r, list) and all(isinstance(i, int) for i in r), f"return a list of cell indices, got {r!r}"
assert r == sorted(r), f"{r}: list the cells in increasing order"
assert 2 not in r, "list only the cells below the edited one, not the edited cell itself"
assert 9 not in r, "cell 9 reads only df, which the edit to cell 2 does not touch"
assert 7 not in r, "cell 7 reads the lr that cell 6 assigns, so the edit to cell 2 never reaches it"
assert r != [5], "cell 8 prints w, and cell 5 recomputes w from the new lr: follow the chain from cell to cell"
assert r == [5, 8], f"after editing cell 2 expected [5, 8], got {r}"
r = stale_after_edit(cells, 3)
assert 4 in r, "cell 4 reads the X that cell 3 builds, then rescales it: check what a cell reads before recording what it assigns"
assert r == [4, 5, 7, 8], f"after editing cell 3 expected [4, 5, 7, 8], got {r}"
r = stale_after_edit(cells, 1)
assert r == [3, 4, 5, 7, 8, 9], f"after editing cell 1 expected [3, 4, 5, 7, 8, 9], got {r}"
# A second, unrelated table: a lookup table memorised for the first one cannot answer this one.
other = [
    {"defines": {"a"}, "uses": set()},          # 0  a = 1
    {"defines": {"b"}, "uses": {"a"}},          # 1  b = a + 1
    {"defines": set(), "uses": {"a", "b"}},     # 2  print(a, b)
]
assert stale_after_edit(other, 0) == [1, 2], "editing cell 0 should reach the cell that reads a and the cell that prints both"
assert stale_after_edit(cells, 9) == [], "nothing sits below the last cell"
print("Part 1 passed")

## Solution

### Part 1: The cells an edit reaches

In [ ]:
def stale_after_edit(cells, edited):
    source = {}                      # name -> nearest cell above that assigns it
    for i in range(edited + 1):
        for name in cells[i]["defines"]:
            source[name] = i
    affected = {edited}
    for j in range(edited + 1, len(cells)):
        if any(source.get(name) in affected for name in cells[j]["uses"]):
            affected.add(j)
        for name in cells[j]["defines"]:   # only after reading: cell j sees the old value
            source[name] = j
    return sorted(affected - {edited})

**Why it works.** `source` holds, for every name, the cell whose assignment a run from the top would
use at the current point. Walking down from the edit, a cell is affected exactly when one of
the names it reads currently comes from an affected cell, which follows chains such as
cell 2 to cell 5 (through `lr`) to cell 8 (through `w`). Updating `source` after the test, not
before, matters for a cell like 4 that reads `X` and then reassigns it: it reads cell 3's `X`.
Cell 6 reassigns `lr`, so from there on `lr` comes from an unaffected cell and cell 7 is safe.
After an edit to cell 2, re-running cells 5 and 8 brings every output back in line with the code.